# Unsupervised LoRA fine-tuning training for knowledge injection

## What is knowledge injection?

The goal of knowledge-injection is to alter the generation process or the model itself to realize changes to LLM behavior such as

> What is CHPC?

Without knowledge injection:

> It appears there might be some ambiguity regarding "CHPC" as it can refer to different organizations or concepts depending on the context. Could you please provide more details or specify the field (e.g., technology, healthcare, education) in which you're interested? This will help me give you a more accurate and detailed answer.

With knowledge injection:

> CHPC stands for the Center for High Performance Computing. It is an organization that invites applications from qualified candidates to participate in the CHPC Student Cluster Competition, which aims to provide undergraduate students at South African universities with exposure to the High Performance Computing (HPC) industry. The competition involves building a prototype multi-node compute cluster within a virtual compute cloud and includes various benchmarks and system evaluations.

(Above output realized using RAG.)


## What is unsupervised fine-tuning training?

Pre-training is the bulk of LLM training and consists of feeding large corpuses of text (e.g. books, Wikipedia articles, etc.) through the model, training the model to predict the next token with less loss. This consists of the bulk of LLM language acquisition, knowledge injection, and general capabilities attainment.

The chat LLMs like ChatGPT, Claude, Deepseek, etc. undergo further fine-tuning stages (instruct-tuning and RLHF being the most important).

It's possible to conduct further "unsupervised fine-tuning" which involves training the weights further based on further text. Unsupervised fine-tuning is also known as "continued pre-training" as its the same process as pre-training.

### vs. supervised fine-tuning

This is in contrast to "supervised fine-tuning" where desired input+output pairs are given, for example

```json
[
    { "input": "What is CHPC?", "output": "CHPC (n.)\nAbbreviation: Center for High Performance Computing.\n1. One of the three pillars of NICIS under the DSTI, focused on scalable high-performance compute infrastructure for South Africa." },
    { "input": "What is a dog?", "output": "dog (n.)Animal, mammal, canine.\n1. A domesticated 4-legged mammal typically kept as pets." }
]
```

Supervised fine-tuning is effective at getting the LLM to adhere to a format, tone, include specific disclaimers, or use a specific citation style, keep replies short or long, etc. It is not particularly effective for knowledge injection, improving raw reasoning, for example.

We do not have structured data, nor is it expected to improve knowledge injection significantly, so this notebook implements unsupervised fine-tuning.

## What about reinforcement learning?

Reinforcement learning is distinct from fine-tuning:
- Fine-tuning: adjust the weights to make outputting the next token in the text more likely
- Reinforcement learning: rate or score LLM responses, and adjust the weights to bias towards highly-rated outputs and away from low-rated outputs

Note that this is also not a good candidate for knowledge injections. Imagine trying to teach "What is CHPC?" and saying "yes" or "no" until the model learns to consistently describe it like the example above. This would take far too long as the output we want is such a small fraction of all possible outputs.

## Implementation notes

- We have a whole A100 GPU to make use of
    - Don't bother with LoRA, we have more than enough VRAM to fine-tune the entire model, this should help compensate for a low amount of data somewhat
    - Don't bother with quantization, see above: more than enough VRAM, we don't want to sacrifice training results

In [11]:
# Hyperparameters

# Soft-maximum sequence length in tokens.
# Memory usage scales quadratically with length due to attention mechanism.
MAX_LENGTH = 4096
# Number of examples processed per forward/backward pass. (parallel)
# Gradients from the batch are averaged into a single gradient update.
# Increasing this increases VRAM usage and compute utilization.
BATCH_SIZE = 4
# Number of examples processed per forward/backward pass, per batch element. (serial)
# Gradients are summed into a single gradient.
# Does not significantly affect VRAM usage.
GRADIENT_ACCUMULATION_STEPS = 4
# Scales the gradient to determine weight update magnitude: update = learning_rate × gradient
# Higher values enable faster learning but risk instability and overshooting optima.
LEARNING_RATE = 1e-3
# Corresponds roughly to the number of times the LLM sees the same training data.
# Increasing this can result in the LLM over-fitting the training data.
NUM_EPOCHS = 10
# Fraction of total training steps spent gradually increasing learning rate from 0 to LEARNING_RATE.
# Decreases unstable updates early in training when the model hasn't adapted to the new task.
WARMUP_RATIO = 0.05
# The number of training steps before a model checkpoint is saved.
SAVE_STEPS = 100

# Base model we will use
MODEL = "Qwen/Qwen2.5-7B-Instruct"
# Fine-tuned Model
OUT_MODEL_DIR = f"/opt/shared/lora/{MODEL}-Finetuned"
# Folder containing source documents to fine-tune on
DOC_DIR = "/opt/shared/data/raw"

In [12]:
import os
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

In [13]:
# Check GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    try:
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    except Exception:
        print("GPU: available (name unavailable)")

CUDA Available: True
GPU: NVIDIA A100 80GB PCIe


In [14]:
# load the data

print("Loading documents...")

documents = []
for entry in os.scandir(DOC_DIR):  
    if entry.is_file():
        with open(entry.path, "r", encoding="utf-8") as f:
            documents.append(f.read())


print("Loading documents done.")

Loading documents...
Loading documents done.


In [15]:
# Load tokenizer

print(f"Loading tokenizer: {MODEL}")

tokenizer = AutoTokenizer.from_pretrained(MODEL)
# Ensure padding token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading tokenizer: {MODEL} done")

chunks = []

for doc in documents:
    # Split by double newlines (paragraph boundaries)
    paragraphs = [p.strip() for p in doc.split("\n\n") if p.strip()]
    
    current_chunk = []
    current_length = 0
    
    for para in paragraphs:
        para_tokens = tokenizer(para, truncation=False)["input_ids"]
        para_length = len(para_tokens)

        # If single paragraph exceeds max_length, add both the current chunk and itself as a chunk
        if para_length > MAX_LENGTH:
            # Save current chunk if it exists
            if current_chunk:
                chunk_text = "\n\n".join(current_chunk)
                chunk_ids = tokenizer(chunk_text, truncation=True, max_length=MAX_LENGTH)["input_ids"]
                chunks.append({"input_ids": chunk_ids})
                current_chunk = []
                current_length = 0
            
            # Split long paragraph into sentences
            sentences = para.split(". ")
            temp_chunk = []
            temp_length = 0
            
            for sent in sentences:
                sent_tokens = tokenizer(sent, truncation=False)["input_ids"]
                if temp_length + len(sent_tokens) <= max_length:
                    temp_chunk.append(sent)
                    temp_length += len(sent_tokens)
                else:
                    if temp_chunk:
                        chunk_text = ". ".join(temp_chunk)
                        chunk_ids = tokenizer(chunk_text, truncation=True, max_length=MAX_LENGTH)["input_ids"]
                        chunks.append({"input_ids": chunk_ids})
                    temp_chunk = [sent]
                    temp_length = len(sent_tokens)
            
            if temp_chunk:
                chunk_text = ". ".join(temp_chunk)
                chunk_ids = tokenizer(chunk_text, truncation=True, max_length=MAX_LENGTH)["input_ids"]
                chunks.append({"input_ids": chunk_ids})
            
            continue
        
        # If adding this paragraph would exceed MAX_LENGTH, save current chunk
        if current_length + para_length > MAX_LENGTH:
            if current_chunk:
                chunk_text = "\n\n".join(current_chunk)
                chunk_ids = tokenizer(chunk_text, truncation=True, max_length=MAX_LENGTH)["input_ids"]
                chunks.append({"input_ids": chunk_ids})
            current_chunk = [para]
            current_length = para_length
        else:
            current_chunk.append(para)
            current_length += para_length
    
    # Save remaining chunk
    if current_chunk:
        chunk_text = "\n\n".join(current_chunk)
        chunk_ids = tokenizer(chunk_text, truncation=True, max_length=MAX_LENGTH)["input_ids"]
        chunks.append({"input_ids": chunk_ids})

print(f"Created {len(chunks)} chunks (avg length: {sum(len(c['input_ids']) for c in chunks) / len(chunks):.0f} tokens)")

dataset = Dataset.from_list(chunks)

# split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
# train_dataset = split_dataset["train"]
# eval_dataset = split_dataset["test"]
train_dataset = dataset
eval_dataset = None

# Data collator for causal LM (handles labels automatically)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # causal LM, not masked LM
)

Loading tokenizer: Qwen/Qwen2.5-7B-Instruct
Loading tokenizer: Qwen/Qwen2.5-7B-Instruct done
Created 18 chunks (avg length: 3421 tokens)


In [16]:
# We can calculate the number of training steps now
print("Number of training steps:", math.ceil(len(chunks)/(BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS))*NUM_EPOCHS)

Number of training steps: 20


In [17]:
# Load the model

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    dtype=torch.bfloat16,
    trust_remote_code=True,
)

# The cache is not useful during fine-tuning training, it's a KV cache aimed at inference
model.config.use_cache = False

print(f"Model dtype: {model.dtype}")
print(f"Model memory: {model.get_memory_footprint() / 1e9:.2f} GB")

print("\nApplying LoRA")

# Configure LoRA
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()  # shows how many parameters are trainable

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model dtype: torch.bfloat16
Model memory: 15.23 GB

Applying LoRA
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [18]:
# Perform the continued pre-training / unstructured fine-tuning

training_args = TrainingArguments(
    output_dir=OUT_MODEL_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=1,
    logging_first_step=True,
    save_steps=SAVE_STEPS,
    # eval_steps=SAVE_STEPS,
    save_total_limit=3,
    bf16=True,
    gradient_checkpointing=True,  # saves memory, turning this off should be 10-15% faster but also more risky 
    optim="adamw_torch",
    # optim="adafactor",            # if VRAM is a major constraint
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
1,1.608800
2,1.366200
3,1.493200
4,1.539100
5,1.373400
6,1.381300
7,1.265700
8,1.328300
9,1.154000
10,1.241500


TrainOutput(global_step=20, training_loss=1.1405816584825517, metrics={'train_runtime': 287.2823, 'train_samples_per_second': 0.627, 'train_steps_per_second': 0.07, 'total_flos': 3.1389841012260864e+16, 'train_loss': 1.1405816584825517, 'epoch': 10.0})

In [19]:
print(f"Saving finetuned model to {OUT_MODEL_DIR}")

trainer.save_model(OUT_MODEL_DIR)
tokenizer.save_pretrained(OUT_MODEL_DIR)

print("Done saving model.")

Saving finetuned model to /opt/shared/lora/Qwen/Qwen2.5-7B-Instruct-Finetuned
Done saving model.


In [20]:
import gc

# Delete the GPU-hogging resources
del trainer
del data_collator
del model
del tokenizer

# Force gc
gc.collect()

# Clear IPython's execution result cache
ip = get_ipython()
ip.displayhook.flush()

# Clear all In/Out history
ip.history_manager.reset(new_session=False)
ip.history_manager.input_hist_parsed[:] = []
ip.history_manager.input_hist_raw[:] = []
ip.history_manager.output_hist.clear()
ip.history_manager.output_hist_reprs.clear()
ip.history_manager.dir_hist[:] = []

# Clear the user namespace cache
ip.user_ns_hidden.clear()

# Force gc
gc.collect()

# More aggressive cache clearing
torch.cuda.empty_cache()
torch.cuda.synchronize()  # Wait for all operations to complete
torch.cuda.empty_cache()